# 03 · Random Forest


Se entrena con junio, se evalúa con septiembre y solo en las localidades habilitadas. Se comparan tres conjuntos de variables (`variables_v2.CONJUNTOS`): `v1` (las del cuaderno 02), `v2` (v1 + tipología, más amenidades, noches mínimas recortadas y estancia mensual) y `v2_geo` (v2 + latitud y longitud).

Además del MdAPE sobre toda la captura se registra `mdape_nuevos`, que es el error en los anuncios que no existían en junio (una simulación del caso real, dado que son anuncios nuevos). 

In [ ]:
from sklearn.ensemble import RandomForestRegressor

import numpy as np
import pandas as pd

from bogoprice import variables_v2
from bogoprice.experimentos import  evaluar, preparar_datos, tabla_resultados
from bogoprice.tracking import iniciar_mlflow

# Tag de autor de MLflow
AUTOR = "Edgar"          
iniciar_mlflow("bogoprice-random-forest")
datos = preparar_datos()
resultados = []

## 1. Conjuntos de variables

El base para los tres tipos de variables

In [ ]:
def rf(**p):
    # Valores base: cada prueba cambia solo uno de ellos
    base = {"n_estimators": 300, "min_samples_leaf": 5, "max_features": 0.5}
    return RandomForestRegressor(n_jobs=-1, random_state=0, **(base | p)), base | p

for version in ["v1", "v2", "v2_geo"]:
    modelo, params = rf()
    m = evaluar(datos, f"rf_{version}", modelo, version, {"modelo": "RandomForest", **params}, AUTOR)[0]
    resultados.append({"corrida": f"rf_{version}", "conjunto": version,
                       "parametro": "base", "valor": "-", **m})
    print(f"{version:7s} MdAPE nuevos = {m['mdape_nuevos']:.2f} %")

## 2. Hiperparámetros (v2 con y sin coordenadas)

Se parte de la configuración base (300 árboles, `min_samples_leaf=5`, `max_features=0.5`) y se cambia un hiperparámetro a la vez, dejando los demás constantes. Así cada corrida muestra el efecto de un solo parámetro.

In [ ]:
# Cada parámetro se prueba con varios valores; los demás quedan en la base
pruebas = {
    "n_estimators": [100, 500],
    "max_depth": [10, 20],
    "max_features": [0.3, 1.0],
    "min_samples_leaf": [1, 10, 20],
    "max_samples": [0.5],
    "min_samples_split": [30],
    "bootstrap": [False],
    "criterion": ["poisson"],
    "ccp_alpha": [0.001],
}

for version in ["v2", "v2_geo"]:
    for parametro, valores in pruebas.items():
        for valor in valores:
            modelo, params = rf(**{parametro: valor})
            nombre = f"rf_{parametro}{valor}_{version}"
            m = evaluar(datos, nombre, modelo, version,
                           {"modelo": "RandomForest", "parametro_variado": parametro, **params}, AUTOR)[0]
            resultados.append({"corrida": nombre, "conjunto": version, "parametro": parametro,
                               "valor": valor, **m})

## Resumen

Resumen de los resultados

In [ ]:
tabla = pd.DataFrame(resultados)
tabla.pivot_table(index=["parametro", "valor"], columns="conjunto", values="mdape_nuevos").round(2)